In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

# Setup

In [2]:

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


# Dataset

In [3]:
from jetnet.datasets import JetNet
from jetnet.datasets.normalisations import FeaturewiseLinearBounded, FeaturewiseLinear

/Users/arjunsharma/development/s25-fermilab-research/src/venv/lib/python3.10/site-packages/coffea/nanoevents/schemas/fcc.py:5: FutureWarning: In version 2025.1.0 (target date: 2024-12-31 11:59:59-06:00), this will be an error.
To raise these warnings as errors (and get stack traces to find out where they're called), run
    import warnings
    warnings.filterwarnings("error", module="coffea.*")
after the first `import coffea` or use `@pytest.mark.filterwarnings("error:::coffea.*")` in pytest.
Issue: coffea.nanoevents.methods.vector will be removed and replaced with scikit-hep vector. Nanoevents schemas internal to coffea will be migrated. Otherwise please consider using that package!.
  from coffea.nanoevents.methods import vector


In [4]:
MASK = True
NUM_PARTICLES = 150
TRAIN_SPLIT = 0.7

In [5]:
feature_maxes = JetNet.fpnd_norm.feature_maxes
if MASK:
    feature_maxes = feature_maxes + [1]

data_args = {
    "jet_type": ["g", "q", "t"],
    "data_dir": "datasets/jetnet",
    "num_particles": NUM_PARTICLES,
    "particle_features": (
        JetNet.ALL_PARTICLE_FEATURES if MASK else JetNet.ALL_PARTICLE_FEATURES[:-1]
    ),
    # The order of the list is preserved in the retrieved data
    "jet_features": ["eta", "pt", "mass", "num_particles", "type"],
    # "particle_normalisation": particle_normalizer,
    "split_fraction": [TRAIN_SPLIT, 1 - TRAIN_SPLIT, 0],
    "download": True
}

In [6]:
from torch.utils.data import DataLoader
X_train = JetNet(**data_args, split="train")
X_test = JetNet(**data_args, split="valid")

In [7]:
len(X_train), len(X_test)

(368113, 157763)

In [8]:
X_train[:][0][:, :, :].shape

torch.Size([368113, 150, 4])

`X_train` consists of 368113 jets; each of which is represented as a tuple with 2 elements:
1. A shape `30 x 4` tensor representing each of the particles in the jet, where the features are in the order of ['etarel', 'phirel', 'ptrel', 'mask']
2. A length `5` tensor consisting of jet features ["eta", "pt", "mass", "num_particles", "type"],

### Cleaning

We convert the particles from relative polar coordinates to absolute Cartesian coordinates. To do this, we use the JetNet `relEtaPhiPt_to_cartesian` utility function, which takes in two parameters:
1. Particle features, where the last axis is $\eta^\text{rel}, phi^\text{rel}, p_\text{T}^\text{rel}$
2. Jet features, where the last axis is $\eta, \phi, p_\text{T}, E/c$

$E/c$ is equivalent to jet mass. Values for the azimuthal angle $\phi$ are not provided for jets in the dataset due to the azimuthal symmetry of the collider system. We therefore provide random $\phi$ values for the jets.

In [9]:
from jetnet.utils import EtaPhiPtE_to_cartesian

def transform_rel_particle_coordinates_to_cartesian(X):
    """
    Transforms relative particle coordinates to absolute Cartesian coordinates using the JetNet relEtaPhiPt_to_cartesian utility function

    Requires X to be a list of length N_jets where each item is a tuple (particle_features, jet_features)
    where particle_features is of shape (n_particles, n_particle_features)
    and jet_features is of length n_jet_features

    Particle features need to start as etarel, phirel, ptrel
    Jet features need to start as eta, pt, mass

    The function generates random phi-values for jets taking into account the azimuthal symmetry of the collider
    """

    particle_polarrel_features = X[:][0][:, :, :3]
    masks = X[:][0][:, :, 3] 
    
    # Phi has to be the second column for the JetNet utility function
    jet_eta = (X[:][1][:, 0]).unsqueeze(1)
    jet_phi_vals = (2 * torch.pi) * torch.rand(len(X)).unsqueeze(1)
    jet_pt_ec = X[:][1][:, 1:3]
    jet_features = torch.concat([jet_eta, jet_phi_vals, jet_pt_ec], dim=-1)

    # Because of issues with the JetNet utility implementation, we do the conversion ourselves
    eta_rel, phi_rel, pt_rel = torch.unbind(particle_polarrel_features, axis=-1)
    Eta, Phi, Pt, _ = torch.unbind(jet_features, axis=-1)

    pt = pt_rel * Pt.unsqueeze(1)
    eta = eta_rel + Eta.unsqueeze(1)
    phi = phi_rel + Phi.unsqueeze(1)
    p0 = pt * torch.cosh(eta)

    stacked = torch.stack([eta, phi, pt, p0], axis=-1)
    cartesian_feats = EtaPhiPtE_to_cartesian(stacked)
    # Return the Cartesian coordinates and the masks
    return torch.cat([cartesian_feats, masks.unsqueeze(-1)], dim=-1)


In [10]:
X_train_particle_transformed = transform_rel_particle_coordinates_to_cartesian(X_train)
X_test_particle_transformed = transform_rel_particle_coordinates_to_cartesian(X_test)

print(X_train_particle_transformed.shape)
print(X_test_particle_transformed.shape)

torch.Size([368113, 150, 5])
torch.Size([157763, 150, 5])


## Models

### Architecture

For testing, we design a simple message-passing Lorentz equivariant network based on LorentzNet

The inputs to each layer are the particle features $x_i$ for $x = 1 \dots N$, where $N$ is the number of particles (30). The message $m_{ij}^l$ between particles $i$ and $j$ in the $l$-th layuer is
$$
m^l_{ij} = \phi_e(\psi(||x_i^l - x_j^l||)), \psi(<x_i^l, x_j^l>))\\
m^l_{ij} = \phi_m(m^l_{ij}) m^l_{ij}
$$

where $\phi_e$ is a neural network, $\psi(\cdot) = \text{sgn} \log (|\cdot| + 1)$, $|| \cdot || $ is the Minkowski norm, and $<\cdot, \cdot>$ is the Minkowski inner product

The updated velocity after the $l$-th step is
$$
x_i^{l+1} = x_i^l + c \sum\limits_{j=1}^N \phi_x(m_{ij}, t') \cdot |x_j^l - x_i^l|
$$

where $\phi_x(m_{ij})$ is a scalar, preserving lorentz equivariance; and $t'$ is the time embedding

## Masking